# Japanese → English S2ST — Colab
GPUランタイムを選び、上から実行します。Driveへのアクセス許可が必要です。
既定は各split 5件・2更新。長期学習は明示設定が必要です。
再接続時も同じDriveパス・repository revision・設定を使ってください。
Colabの寿命は保証されません。自動再接続・制限回避は行いません。
詳細: [docs/COLAB.md](../docs/COLAB.md)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, subprocess, json
# 自分の保存先へ変更。コーパスの下に出力を置かないこと。
CORPUS = Path('/content/drive/MyDrive/ja-en-direct-s2st-corpus')
PERSISTENT = Path('/content/drive/MyDrive/s2st-benchmark')
DATA = PERSISTENT / 'smoke-data'
REPO = Path('/content/ja-en-direct-s2st-benchmark')
PERSISTENT.mkdir(parents=True, exist_ok=True)
def run(*args):
    subprocess.run([str(a) for a in args], check=True)
assert not PERSISTENT.resolve().is_relative_to(CORPUS.resolve())

## コードの取得
初回のGit revisionをDriveへ記録し、以後は同じrevisionを使用します。コード更新時は新しい実験保存先を使ってください。

In [ ]:
revision_file = PERSISTENT / 'repository-revision.txt'
if not REPO.exists():
    run('git', 'clone', '--recurse-submodules',
        'https://github.com/Yaaamashiro/ja-en-direct-s2st-benchmark.git', REPO)
if revision_file.exists():
    revision = revision_file.read_text().strip()
    run('git', '-C', REPO, 'fetch', 'origin', revision)
    run('git', '-C', REPO, 'checkout', '--detach', revision)
else:
    revision = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
    revision_file.write_text(revision)
os.chdir(REPO)
print('Pinned repository:', revision)

## ランタイムの構築（VMごとに必要）
システムのtorchを置換せず、独立Python 3.13.7環境へ固定依存をインストールします。初回は時間がかかります。旧版の実験保存先は使わず、新しい保存先・ランタイムで始めてください。

In [ ]:
run('python', REPO / 'scripts/colab/setup.py')
PYTHON = Path('/content/s2st-runtime/venv/bin/python')
os.environ.update({
    'CORPUS_ROOT': str(CORPUS), 'EXPERIMENT_DATA_ROOT': str(DATA),
    'RUNS_ROOT': str(PERSISTENT / 'runs'), 'CACHE_ROOT': str(PERSISTENT / 'cache'),
    'S2ST_CONFIG_ROOT': str(REPO / 'configs'),
    'PYTHONPATH': '/content/s2st-runtime/fairseq',
    'PIP_CONSTRAINT': str(REPO / 'requirements/colab313.txt'),
    'ESPEAK_DATA_PATH': '/content/s2st-runtime/espeak/share/espeak-ng-data',
    'PATH': '/content/s2st-runtime/espeak/bin:' + os.environ['PATH'],
    'LD_LIBRARY_PATH': '/content/s2st-runtime/espeak/lib:' + os.environ.get('LD_LIBRARY_PATH', ''),
})
def cli(*args):
    run(PYTHON, '-m', 'direct_s2st.cli', *args)
run('/content/s2st-runtime/espeak/bin/espeak-ng', '--version')

## コーパス受け入れと前処理
既存の分割を保持します。辞書にない文字などはエラーとして確認してください。切断後は同じセルを再実行できます。
本学習用データは別のDATAパスで準備し、limitとprofileを意図的に変更します。

In [ ]:
cli('corpus', 'import', '--profile', 'smoke', '--limit', '5', '--resume')
cli('corpus', 'validate', '--profile', 'smoke')

In [ ]:
# TT2用。S2UTだけならこのセルは省略可。
cli('translatotron2', 'phonemize', '--profile', 'smoke', '--resume')
cli('translatotron2', 'prepare', '--profile', 'smoke', '--resume')
cli('translatotron2', 'validate', '--profile', 'smoke')

In [ ]:
# S2UT / unit vocoder用。TT2だけならこのセルは省略可。
cli('s2ut', 'fetch-artifacts', '--profile', 'smoke', '--resume')
cli('s2ut', 'extract-units', '--profile', 'smoke', '--limit', '5', '--resume')
cli('s2ut', 'prepare', '--profile', 'smoke', '--resume')
cli('s2ut', 'validate', '--profile', 'smoke')

## 学習設定を固定
BACKENDを tt2 / s2ut / unit / mel から選択します。方式ごとにRUN_NAMEを分けます。
TT2のsmokeモデルは接続検証用。本学習レシピ・学習率等は新しいrunを開始する前に明示設定してください。

In [ ]:
BACKEND = 'tt2'
RUN_NAME = 'tt2-smoke'
MODEL_SIZE = 'smoke'
# 初回smoke確認後、新しいRUN_NAMEでgpu80を選ぶ。
PERFORMANCE = 'smoke'  # 'gpu80': TT2 batch8 / vocoder batch16 / workers4
BATCH_SIZE = None      # None=プリセット。S2UTはMAX_TOKENSで調整
NUM_WORKERS = None
PREFETCH_FACTOR = 2    # TT2/vocoderのCPU先読み（workerあたりのサンプル数）
MAX_TOKENS = None      # S2UTのみ。gpu80既定20000
UPDATE_FREQ = 1        # TT2/S2UTのみ。勾配蓄積、vocoderは1
SAVE_INTERVAL = None  # ローカルcheckpoint更新間隔。gpu80既定50
options = dict(performance=PERFORMANCE, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
               prefetch_factor=PREFETCH_FACTOR, max_tokens=MAX_TOKENS,
               update_freq=UPDATE_FREQ, save_interval=SAVE_INTERVAL)
assert RUN_NAME and all(c.isalnum() or c in '-_' for c in RUN_NAME)
CONFIG = PERSISTENT / (RUN_NAME + '.json')
# 設定生成も固定Python環境で実行する。
builder = """import json, sys
from pathlib import Path
from direct_s2st.colab import make_config
cfg = make_config(sys.argv[1], sys.argv[2], '/content/s2st-runtime/environment.json', sys.argv[4], sys.argv[5], **json.loads(sys.argv[6]))
path = Path(sys.argv[3])
if path.exists():
    assert json.loads(path.read_text()) == cfg, 'Config changed: use a new run name'
else:
    path.write_text(json.dumps(cfg, indent=2))
print(json.dumps(cfg, indent=2))
"""
run(PYTHON, '-c', builder, REPO, DATA, CONFIG, BACKEND, MODEL_SIZE, json.dumps(options))

## 学習・Drive保存・再開
1区間が完了するごとにcheckpointとログをDriveへコピー・検証します。
再実行ではRESUME=Trueに変更してください。総更新数は追加分ではなく累計です。
時間予算は学習subprocessの上限で、保存・初期化時間は別です。余裕を残してください。
突然切れた区間は再計算になります。少なくとも1区間の保存が完了するまで再開点はありません。

In [ ]:
TOTAL_UPDATES = 2
CHUNK_UPDATES = 100 if PERFORMANCE == 'gpu80' else 1  # Drive同期・プロセス再起動間隔
SESSION_SECONDS = 3600
RESUME = False
CONFIRM_TRAINING = False  # 累計10更新を超える場合のみ、意図を確認してTrue
args = [PYTHON, '-m', 'direct_s2st.colab', '--config', CONFIG,
        '--work-root', '/content/s2st-work/' + RUN_NAME,
        '--backup-root', PERSISTENT / 'checkpoints' / RUN_NAME,
        '--total-updates', str(TOTAL_UPDATES), '--chunk-updates', str(CHUNK_UPDATES),
        '--session-seconds', str(SESSION_SECONDS)]
if RESUME:
    args.append('--resume')
if CONFIRM_TRAINING:
    args.append('--confirm-training')
run(*args)

## 次の段階
このノートブックは前処理と学習の保存・再開を扱います。完了はE2E成功を意味しません。
推論・vocoder・評価は既存CLIを使います。checkpointはローカルwork-rootに復元されるため、推論設定の参照先を合わせてください。
Cascade・評価の追加依存、詳細、制約はdocs/COLAB.mdを参照してください。
Driveの世代バックアップは自動削除しません。容量を確認し、正常な最新世代を残した上で不要な古い世代を手動整理してください。